# 9.1 Difference equations

In a digital signal processing context, a {vocab}`difference equation` defines a filter by expressing each output sample as a formula in terms of the input samples. It is the most direct, hands-on way to specify a filter, and it translates immediately into code.

## A first example

Consider the difference equation

$$\purple{y[n]} = \blue{x[n]} + \green{x[n-1]}.$$

Each output sample is the current input sample plus a _delayed copy_ of the input. The term $\green{x[n-1]}$ is the input shifted one sample later in time, in other words the value the input had one sample ago. Referring to a delayed copy of a signal like this, written $\green{x[n-d]}$ for a delay of $d$ samples, is the fundamental building block of every filter in this chapter, so make sure the idea feels natural before moving on.

Let us feed the filter a simple square wave with a ten-sample period, $x[n] = [1, 1, 1, 1, 1, -1, -1, -1, -1, -1, \ldots]$. To evaluate $x[n-1]$ at the very start, we need $x[-1]$, which lies before the signal begins. Throughout this chapter we adopt the standard convention that a signal is _zero_ at any index outside its defined range: $x[n] = 0$ for $n < 0$. The very first output sample therefore sees a delayed copy that is still "warming up" from zero.

:::{figure}
![Three stacked stem plots over sample indices 0 to 31. Top (blue): the input square wave x[n], five samples at plus one then five at minus one, repeating. Middle (red): x[n-1], the same square wave shifted one sample to the right, with the first sample (shaded) equal to zero. Bottom (purple): the sum y[n], which reaches plus or minus two across the flat stretches and steps through zero at each transition, after a one-sample warm-up.](./assets/fig-diffeq-lowpass.png)

The filter $\purple{y[n]} = \blue{x[n]} + \green{x[n-1]}$ applied to a square wave. The delayed copy $\green{x[n-1]}$ (green) is the input shifted right by one sample, with a zero assumed before $n = 0$ (shaded). Summing it with $\blue{x[n]}$ gives $\purple{y[n]}$ (purple).
:::

Comparing $y[n]$ to $x[n]$, a few things stand out:

1. The output has a larger _peak amplitude_, reaching $\pm 2$ where the two copies agree.
1. It has a slightly different _shape_, with the square wave's abrupt transitions softened into a step.
1. There is a brief "warm-up" at the very start.

Softening abrupt transitions is a hint that this filter smooths the signal by attenuating its high frequencies, which we will confirm later. You can experiment with this filter in code, including listening to the input and output, in the following example:

In [ ]:
# hide
import numpy as np
import matplotlib.pyplot as plt
import pyquist as pq


def plot_filter_input_output(x, y):
    """Stem-plot an input (blue) above its filtered output (purple)."""
    fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
    for ax, data, label, color in zip(axes, [x, y], [r"$x[n]$", r"$y[n]$"], ["C0", "C4"]):
        ml, sl, bl = ax.stem(range(len(data)), data)
        plt.setp(ml, color=color, markersize=5)
        plt.setp(sl, color=color)
        plt.setp(bl, color="0.7", linewidth=1.0)
        ax.set_ylabel(label, color=color, fontsize=13)
        ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel(r"$n$")
    plt.tight_layout()
    plt.show()

In [ ]:
# The difference equation y[n] = x[n] + x[n-1], applied to a square wave.
# Edit the equation in the loop and re-run to explore other filters!
f_s = 44100
N = f_s                                       # one second of audio
n = np.arange(N)
x = np.where((n // 5) % 2 == 0, 1.0, -1.0)    # square wave, 10-sample period

y = np.zeros(N)
for n in range(N):
    if n - 1 < 0:                             # samples before the start are 0
        y[n] = x[n]
    else:
        y[n] = x[n] + x[n - 1]

# The same computation, vectorized:
# y = x + np.concatenate([np.zeros(1), x])[:N]

plot_filter_input_output(x[:40], y[:40])      # show the first 40 samples

In [ ]:
# Listen to the input and the filtered output (played at reduced gain).
pq.play(pq.Audio(0.4 * x, f_s))
pq.play(pq.Audio(0.4 * y, f_s))

## A second example

Now consider a closely related difference equation that _subtracts_ the delayed copy instead of adding it, and scales both terms by one half:

$$\purple{y[n]} = \tfrac{1}{2}\blue{x[n]} - \tfrac{1}{2}\green{x[n-1]}.$$

:::{figure}
![Three stacked stem plots over sample indices 0 to 31. Top (blue): one half times x[n], a square wave between plus and minus one half. Middle (red): minus one half times x[n-1], the inverted square wave delayed by one sample, with the first sample shaded as a warm-up zero. Bottom (purple): the difference y[n], which is zero across the flat stretches of the square wave and spikes to plus or minus one only at the transitions.](./assets/fig-diffeq-highpass.png)

The filter $\purple{y[n]} = \tfrac{1}{2}\blue{x[n]} - \tfrac{1}{2}\green{x[n-1]}$ applied to the same square wave. Subtracting a delayed copy leaves the output zero wherever the input is constant and produces a spike only at each transition.
:::

The behavior has both differences and similarities compared with the previous example:

1. The one-half factors keep the amplitude in check.
1. Subtracting a delayed copy makes the filter respond only to _changes_ in the input, so the output is zero across the flat stretches and spikes at the transitions.
1. There is again a brief warm-up.

Responding only to change, and ignoring the steady stretches, is a hint that this filter discards low frequencies and keeps the high ones. This filter is, in effect, a crude edge detector.

## What do these filters do to sound?

Difference equations are trivial to implement, but as the two examples show, their effect can be hard to predict just by reading the formula. The clearest way to build intuition is to _listen_. Below are the two filters applied to an audible square-wave tone (a richer square than our ten-sample toy, so many harmonics are in play):

:::{audio-list}
{audio}`Input square wave $x[n]$ <./assets/audio-diffeq-input.wav>`

{audio}`$y_1[n] = x[n] + x[n-1]$ <./assets/audio-diffeq-y1.wav>`

{audio}`$y_2[n] = \frac{1}{2}x[n] - \frac{1}{2}x[n-1]$ <./assets/audio-diffeq-y2.wav>`

By ear, the first filter sounds darker and mellower, the second brighter and thinner.
:::

We can see this directly by passing _white noise_ (which contains every frequency in equal measure) through each filter and plotting the amplitude spectrum of the result. Because the input is spectrally flat, the output spectrum traces out the filter's own frequency response. The two are near-mirror images of each other:

:::{figure}
![Two amplitude spectra over frequency from 0 to f_s over 2, each measured from noise passed through a filter, so both are speckled with measurement noise. One curve, labeled y1, starts high at DC and falls to zero at the Nyquist frequency, a low-pass. The other, labeled y2, starts at zero at DC and rises to its maximum at Nyquist, a high-pass. The two cross near f_s over 4.](./assets/fig-diffeq-responses.png)

The amplitude spectrum of white noise after passing through each filter. Since the input noise is spectrally flat, each output spectrum reveals that filter's frequency response. The first, $y_1$ (a _sum_ of a signal and its delayed copy), passes low frequencies and rolls off the highs: a **low-pass**. The second, $y_2$ (a _difference_), does the reverse: a **high-pass**.
:::

So the sum acts as a low-pass and the difference as a high-pass. We reached both conclusions by ear and by eye, with no theory at all. Over the rest of the chapter we build up several more _perspectives_ on filters like these, each revealing a different facet of how they work.

(sec-convolution)=